# WebSocket stream

Required environment variables: `EXCHANGEOS_ASSET_ID`.

Optional for private order updates: `EXCHANGEOS_API_KEY`, `EXCHANGEOS_API_SECRET`, `EXCHANGEOS_API_PASSPHRASE`, `EXCHANGEOS_VAULT_ADDRESS`.

WebSocket messages are delivered to the callback as raw JSON strings.

In [ ]:
import asyncio
import os

from exchangeos_sdk import Credentials, ExchangeOSClient

credential_names = ("EXCHANGEOS_API_KEY", "EXCHANGEOS_API_SECRET", "EXCHANGEOS_API_PASSPHRASE")
credentials = (
    Credentials(
        api_key=os.environ["EXCHANGEOS_API_KEY"],
        secret_key=os.environ["EXCHANGEOS_API_SECRET"],
        passphrase=os.environ["EXCHANGEOS_API_PASSPHRASE"],
    )
    if all(os.getenv(name) for name in credential_names)
    else None
)

def on_message(raw_json: str) -> None:
    print(raw_json)

In [ ]:
async def stream_for(seconds: float = 60) -> None:
    client = ExchangeOSClient(auth=credentials)
    try:
        async with client.websocket(message_callback=on_message) as ws:
            await ws.subscribe_public(
                "exchangeos-tickers",
                asset_id=os.environ["EXCHANGEOS_ASSET_ID"],
            )
            if credentials:
                await ws.subscribe_private(
                    "exchangeos-orders",
                    vault_address=os.getenv("EXCHANGEOS_VAULT_ADDRESS"),
                )
            else:
                print("Streaming public tickers only; set API credentials to add private order updates.")
            await asyncio.sleep(seconds)
    finally:
        client.close()

await stream_for()